# 04 — Feature Engineering

## Objetivo
Preparar el dataset final para modelado ML: seleccionar, transformar y unir las variables
de las bases 2, 3 y 5 del Relevamiento Anual 2024 en una sola tabla lista para entrenar
un modelo de clasificación de riesgo de abandono escolar.

## Fuentes de entrada
| Base | Archivo | Filas | Columnas | Contenido principal |
|------|---------|-------|----------|---------------------|
| Base 2 | `base2_matricula.csv` | 1.216 | 104 | Matrícula, repetidores, sobreedad, secciones |
| Base 3 | `base3_trayectoria.csv` | 1.209 | 291 | Flujo escolar por grado (entrados, salidos, promovidos) + tasa_abandono |
| Base 5 | `base5_caracteristicas.csv` | 1.216 | 66 | Infraestructura, equipamiento, con

In [1]:
# === CELDA 1: Cargar librerías y bases limpias ===
import pandas as pd
import numpy as np

BASE = '..'

base2 = pd.read_csv(f'{BASE}/data/clean/base2_matricula.csv')
base3 = pd.read_csv(f'{BASE}/data/clean/base3_trayectoria.csv')
base5 = pd.read_csv(f'{BASE}/data/clean/base5_caracteristicas.csv')

print(f'Base 2: {base2.shape}')
print(f'Base 3: {base3.shape}')
print(f'Base 5: {base5.shape}')

Base 2: (1216, 104)
Base 3: (1209, 291)
Base 5: (1216, 66)


## Paso 2 — Exploración de Base 5 (infraestructura y equipamiento)

Los valores de base5 son **conteos**: cuántas escuelas del segmento tienen cada característica.
La columna `localizacion` es el total de escuelas del segmento (el denominador).
Los blancos son strings vacíos, no NaN — hay que limpiarlos antes de operar.

In [2]:
# === CELDA 2: Diagnóstico rápido de base5 ===

# Convertir blancos a NaN y todo a numérico (excepto las 4 columnas clave)
clave = ['provincia', 'departamento', 'sector', 'ambito']
cols_num = [c for c in base5.columns if c not in clave]

for col in cols_num:
    base5[col] = pd.to_numeric(base5[col], errors='coerce')

# Resumen: % de NaN por columna
pct_nan = (base5[cols_num].isna().sum() / len(base5) * 100).round(1)
print("=== Columnas con más de 50% NaN ===")
print(pct_nan[pct_nan > 50].sort_values(ascending=False).to_string())
print(f"\n=== Columnas con 0% NaN: {(pct_nan == 0).sum()} de {len(cols_num)} ===")
print(f"\n=== Localizacion (total escuelas por segmento) ===")
print(base5['localizacion'].describe().round(1))

=== Columnas con más de 50% NaN ===
equipamientobibliotecaequiporeceptorderadioamfm         100.0
equipamientoestablecimientoequiporeceptorderadioamfm    100.0
electricidadgeneradoreolico                              97.6
electricidadgeneradorhidraulico                          97.2
subvencionestataljardinmaternalsi                        87.7
subvencionestatalsnusi                                   85.0
electricidadotro                                         79.6
subvencionestataljardindeinfantessi                      76.4
subvencionestatalprimariasi                              75.7
subvencionestatalsecundariasi                            74.8
sistemadegestionescolarencargadoporelestablecimiento     74.1
electricidadgrupoelectrogeno                             73.4
electricidadpanelfotovoltaicosolar                       72.3
sistemadegestionescolardesarrolladoporterceros           70.6
sistemadegestionescolarplanilladecalculo                 68.1
equipamientobibliotecaservidorpara

## Paso 3 — Construcción de features de Base 5

Estrategia: descartar columnas con >50% NaN, agrupar las restantes en bloques temáticos
y crear una proporción resumen por bloque (cantidad / localizacion).

Los NaN en columnas retenidas se interpretan como 0 (ninguna escuela del segmento
tiene esa característica).

In [3]:
# === CELDA 3: Descartar columnas >50% NaN y ver qué queda ===

# Columnas a descartar (>50% NaN)
cols_descartar = pct_nan[pct_nan > 50].index.tolist()
print(f"Columnas descartadas: {len(cols_descartar)}")

# Columnas numéricas que quedan (sin las 4 clave)
cols_utiles = [c for c in cols_num if c not in cols_descartar]
print(f"Columnas útiles restantes: {len(cols_utiles)}")
print()

# Mostrarlas para revisarlas juntos
for c in cols_utiles:
    pct = pct_nan[c]
    print(f"  {c:55s}  NaN: {pct}%")

Columnas descartadas: 16
Columnas útiles restantes: 46

  localizacion                                             NaN: 0.0%
  electricidadsi                                           NaN: 0.0%
  electricidadredpublica                                   NaN: 0.2%
  equipamientoestablecimientotelevisor                     NaN: 0.1%
  equipamientoestablecimientosistemamultimediaocanon       NaN: 0.4%
  equipamientoestablecimientoscanner                       NaN: 1.7%
  equipamientoestablecimientowebcam                        NaN: 7.3%
  equipamientoestablecimientoreproductordecd               NaN: 5.8%
  equipamientoestablecimientoreproductordedvd              NaN: 1.3%
  equipamientoestablecimientoimpresora                     NaN: 0.2%
  equipamientoestablecimientoequipoemisorderadioamfm       NaN: 4.2%
  equipamientoestablecimientoservidorparausoescolar        NaN: 14.3%
  equipamientoestablecimientoimpresora3d                   NaN: 30.9%
  equipamientoestablecimientoequipodesonido  

## Construcción de 8 features resumen de Base 5

Cada feature es una proporción: cantidad de escuelas con la característica / total de escuelas del segmento.
Los NaN se interpretan como 0 (ninguna escuela tiene esa característica).

In [4]:
# === CELDA 4: Construir features de base5 ===

# Rellenar NaN con 0 en columnas numéricas (NaN = ninguna escuela tiene eso)
base5[cols_utiles] = base5[cols_utiles].fillna(0)

loc = base5['localizacion']

# 1. Proporción con electricidad
base5['prop_electricidad'] = base5['electricidadsi'] / loc

# 2. Proporción promedio de equipamiento del establecimiento
equip_est = [c for c in cols_utiles if c.startswith('equipamientoestablecimiento')]
base5['prop_equip_establecimiento'] = base5[equip_est].div(loc, axis=0).mean(axis=1)

# 3. Proporción promedio de equipamiento de biblioteca
equip_bib = [c for c in cols_utiles if c.startswith('equipamientobiblioteca')]
base5['prop_equip_biblioteca'] = base5[equip_bib].div(loc, axis=0).mean(axis=1)

# 4. Proporción con internet gratuito del Estado
base5['prop_internet_estatal'] = base5['internettipodeserviciogratuitoestado'] / loc

# 5. Proporción con conexión en aulas
base5['prop_conexion_aulas'] = base5['espaciosconconexionenlasaulas'] / loc

# 6. Proporción SIN cooperadora
base5['prop_sin_cooperadora'] = base5['cooperadorano'] / loc

# 7. Proporción con laboratorio de informática
base5['prop_laboratorio'] = base5['disponedesalaolaboratoriodeinformaticasi'] / loc

# 8. Proporción con biblioteca
base5['prop_biblioteca'] = base5['bibliotecadisponedealmenosunasi'] / loc

# Verificar
features_b5 = ['prop_electricidad', 'prop_equip_establecimiento', 'prop_equip_biblioteca',
               'prop_internet_estatal', 'prop_conexion_aulas', 'prop_sin_cooperadora',
               'prop_laboratorio', 'prop_biblioteca']

print(base5[features_b5].describe().round(3).to_string())

       prop_electricidad  prop_equip_establecimiento  prop_equip_biblioteca  prop_internet_estatal  prop_conexion_aulas  prop_sin_cooperadora  prop_laboratorio  prop_biblioteca
count           1216.000                    1216.000               1216.000               1216.000             1216.000              1216.000          1216.000         1216.000
mean               0.988                       0.444                  0.131                  0.522                0.583                 0.505             0.345            0.685
std                0.045                       0.117                  0.086                  0.348                0.284                 0.330             0.198            0.187
min                0.226                       0.038                  0.000                  0.000                0.000                 0.000             0.000            0.000
25%                1.000                       0.361                  0.069                  0.167                0

## Features de Base 2 (matrícula) y Base 3 (trayectoria)

Base 2 aporta: tamaño del segmento, sobreedad, repetidores, secciones.
Base 3 aporta: la tasa_abandono (target) y tasas de flujo por grado.
Misma lógica: convertir conteos en proporciones para que sean comparables.

In [14]:
# === CELDA 6: Features de Base 2 (versión final — 4 features) ===

for col in base2.columns[4:]:
    base2[col] = pd.to_numeric(base2[col], errors='coerce')
base2.fillna(0, inplace=True)

grados_mat = ['_1','_2','_3','_4','_5','_6','_7','_8','_9','_10','_11','_12','_1314','_20']
grados_rep = ['r_1','r_2','r_3','r_4','r_5','r_6','r_7','r_8','r_9','r_10','r_11','r_12','r_1314','r_20']
grados_sob = ['s_1','s_2','s_3','s_4','s_5','s_6','s_7','s_8','s_9','s_10','s_11','s_12','s_1314']
grados_sec = ['_sec1','_sec2','_sec3','_sec4','_sec5','_sec6','_sec7','_sec8','_sec9','_sec10','_sec11','_sec12','_sec1314','_sec20']

# 1. Matrícula total
base2['matricula_total'] = base2[grados_mat].sum(axis=1)

# 2. Tasa de repitencia
base2['tasa_repitencia'] = base2[grados_rep].sum(axis=1) / base2['matricula_total'].replace(0, np.nan)

# 3. Tasa de sobreedad
base2['tasa_sobreedad'] = base2[grados_sob].sum(axis=1) / base2['matricula_total'].replace(0, np.nan)

# 4. Ratio alumnos por sección
base2['ratio_alumnos_seccion'] = base2['matricula_total'] / base2[grados_sec].sum(axis=1).replace(0, np.nan)

# Verificar
features_b2 = ['matricula_total', 'tasa_repitencia', 'tasa_sobreedad', 'ratio_alumnos_seccion']
print(base2[features_b2].describe().round(3).to_string())

       matricula_total  tasa_repitencia  tasa_sobreedad  ratio_alumnos_seccion
count         1216.000         1207.000        1207.000               1204.000
mean          7225.601            0.043           0.123                 22.449
std          15340.313            0.031           0.079                  8.741
min              0.000            0.000           0.000                  5.180
25%           1021.000            0.018           0.062                 18.998
50%           2290.500            0.040           0.115                 21.771
75%           6240.000            0.060           0.168                 25.125
max         222438.000            0.285           0.521                185.000


## Features de Base 3 — solo target

De base3 usamos únicamente `tasa_abandono` para construir el target binario.
No usamos otras variables de flujo (promovidos, salidos, entrados) porque generarían
data leakage: están calculadas a partir del mismo dato que el target.

In [15]:
# === CELDA 7: Verificar tasa_abandono de base3 ===
print(f"Shape base3: {base3.shape}")
print(f"NaN en tasa_abandono: {base3['tasa_abandono'].isna().sum()}")
print(f"\n{base3['tasa_abandono'].describe().round(3)}")

Shape base3: (1209, 291)
NaN en tasa_abandono: 1

count    1208.000
mean        0.856
std         1.718
min         0.000
25%         0.097
50%         0.389
75%         1.197
max        43.333
Name: tasa_abandono, dtype: float64


## Paso 4 — Definición del target binario

Necesitamos un umbral para convertir `tasa_abandono` en `abandono_alto` (1/0).
El umbral debe ser: estadísticamente razonable y operacionalmente útil
(que la cantidad de segmentos "alto riesgo" sea suficiente para entrenar el modelo
pero realista para priorizar intervenciones).

In [16]:
# === CELDA 8: Explorar umbrales posibles ===
ta = base3['tasa_abandono'].dropna()

# Ver percentiles clave
print("=== Percentiles de tasa_abandono ===")
for p in [50, 60, 70, 75, 80, 85, 90, 95]:
    valor = ta.quantile(p/100)
    n_alto = (ta > valor).sum()
    pct_alto = (n_alto / len(ta) * 100)
    print(f"  Percentil {p:2d}: {valor:6.3f}%  →  {n_alto:4d} segmentos alto riesgo ({pct_alto:.1f}%)")

=== Percentiles de tasa_abandono ===
  Percentil 50:  0.389%  →   604 segmentos alto riesgo (50.0%)
  Percentil 60:  0.598%  →   483 segmentos alto riesgo (40.0%)
  Percentil 70:  0.950%  →   363 segmentos alto riesgo (30.0%)
  Percentil 75:  1.197%  →   302 segmentos alto riesgo (25.0%)
  Percentil 80:  1.407%  →   242 segmentos alto riesgo (20.0%)
  Percentil 85:  1.705%  →   182 segmentos alto riesgo (15.1%)
  Percentil 90:  2.186%  →   121 segmentos alto riesgo (10.0%)
  Percentil 95:  2.894%  →    61 segmentos alto riesgo (5.0%)


## Paso 4 — Creación del target binario

Umbral elegido: percentil 75 (tasa_abandono > 1.2%).
Genera la columna `abandono_alto`: 1 = alto riesgo, 0 = bajo riesgo.

In [17]:
# === CELDA 9: Crear target binario ===
UMBRAL = 1.2  # Percentil 75

base3['abandono_alto'] = (base3['tasa_abandono'] > UMBRAL).astype(int)

print(f"Umbral: {UMBRAL}%")
print(f"\n=== Distribución del target ===")
print(base3['abandono_alto'].value_counts())
print(f"\n0 = bajo riesgo: {(base3['abandono_alto']==0).sum()} ({(base3['abandono_alto']==0).mean()*100:.1f}%)")
print(f"1 = alto riesgo: {(base3['abandono_alto']==1).sum()} ({(base3['abandono_alto']==1).mean()*100:.1f}%)")
print(f"NaN (sin dato):  {base3['abandono_alto'].isna().sum()}")

Umbral: 1.2%

=== Distribución del target ===
abandono_alto
0    908
1    301
Name: count, dtype: int64

0 = bajo riesgo: 908 (75.1%)
1 = alto riesgo: 301 (24.9%)
NaN (sin dato):  0


## Paso 5 — Merge de las 3 bases

Unimos Base 2, Base 3 y Base 5 por la clave de segmento: provincia × departamento × sector × ámbito.
Usamos inner join: solo quedan los segmentos que existen en las 3 bases.

In [18]:
# === CELDA 10: Merge de las 3 bases ===
clave = ['provincia', 'departamento', 'sector', 'ambito']

# Seleccionar solo las columnas que necesitamos de cada base
cols_b2 = clave + features_b2
cols_b3 = clave + ['tasa_abandono', 'abandono_alto']
cols_b5 = clave + features_b5

# Merge: Base2 + Base3
df = base2[cols_b2].merge(base3[cols_b3], on=clave, how='inner')
print(f"Después de merge B2 + B3: {df.shape}")

# Merge: + Base5
df = df.merge(base5[cols_b5], on=clave, how='inner')
print(f"Después de merge + B5:    {df.shape}")

print(f"\nColumnas finales: {df.shape[1]}")
print(f"NaN totales: {df.isna().sum().sum()}")
print(f"\n{df.head(3).to_string()}")

Después de merge B2 + B3: (1207, 10)
Después de merge + B5:    (1207, 18)

Columnas finales: 18
NaN totales: 3

      provincia departamento   sector  ambito  matricula_total  tasa_repitencia  tasa_sobreedad  ratio_alumnos_seccion  tasa_abandono  abandono_alto  prop_electricidad  prop_equip_establecimiento  prop_equip_biblioteca  prop_internet_estatal  prop_conexion_aulas  prop_sin_cooperadora  prop_laboratorio  prop_biblioteca
0  Buenos Aires   25 DE MAYO  Estatal   Rural           1234.0         0.038898        0.100486              17.380282       1.033386              0                1.0                    0.396259               0.112245               0.836735             0.897959              0.122449          0.204082         0.510204
1  Buenos Aires   25 DE MAYO  Estatal  Urbano           4086.0         0.078561        0.142682              20.636364       0.778589              0                1.0                    0.424419               0.095930               0.790698       

## Paso 6 — Resolución de NaN y exportación

3 segmentos tienen NaN en `ratio_alumnos_seccion` (0 secciones registradas, probable error de fuente).
Se rellenan con la mediana del ratio. Después se guarda el dataset final en `data/ml/`.

In [20]:
# === CELDA 11: Resolver NaN y guardar dataset ===

# Rellenar 3 NaN de ratio_alumnos_seccion con la mediana
mediana_ratio = df['ratio_alumnos_seccion'].median()
df['ratio_alumnos_seccion'] = df['ratio_alumnos_seccion'].fillna(mediana_ratio)

print(f"NaN restantes: {df.isna().sum().sum()}")
print(f"Mediana usada para ratio_alumnos_seccion: {mediana_ratio:.2f}")
print(f"Shape final: {df.shape}")

NaN restantes: 0
Mediana usada para ratio_alumnos_seccion: 21.77
Shape final: (1207, 18)


## Exportación del dataset final

Dataset listo para modelado: 1.207 segmentos × 18 columnas.
Se guarda en `data/ml/dataset_modelo.csv`, separado de los datos originales.

**Composición:**
- 4 columnas clave (provincia, departamento, sector, ámbito)
- 4 features de Base 2 (matrícula, repitencia, sobreedad, ratio alumnos/sección)
- 8 features de Base 5 (electricidad, equipamiento, conectividad, organización)
- 1 target binario (`abandono_alto`: 1 = alto riesgo, 0 = bajo riesgo)
- 1 columna de referencia (`tasa_abandono`: valor original continuo)

In [21]:
# === CELDA 12: Guardar dataset final ===
import os
os.makedirs(f'{BASE}/data/ml', exist_ok=True)

df.to_csv(f'{BASE}/data/ml/dataset_modelo.csv', index=False)

print(f"Guardado en data/ml/dataset_modelo.csv")
print(f"Shape: {df.shape}")
print(f"Target: {df['abandono_alto'].value_counts().to_dict()}")
print(f"\nColumnas:")
for c in df.columns:
    print(f"  {c}")

Guardado en data/ml/dataset_modelo.csv
Shape: (1207, 18)
Target: {0: 906, 1: 301}

Columnas:
  provincia
  departamento
  sector
  ambito
  matricula_total
  tasa_repitencia
  tasa_sobreedad
  ratio_alumnos_seccion
  tasa_abandono
  abandono_alto
  prop_electricidad
  prop_equip_establecimiento
  prop_equip_biblioteca
  prop_internet_estatal
  prop_conexion_aulas
  prop_sin_cooperadora
  prop_laboratorio
  prop_biblioteca
